# 02 - Run Server + Cloudflare Tunnel

Notebook nay dung moi session Colab de chay server va lay URL cho client.
Yeu cau: da chay clone_repo.ipynb truoc do.

Neu local venv khong ton tai (do runtime moi), notebook se tu goi bootstrap script de tao lai.
Luu y: notebook 01 va notebook 02 co the dang o hai runtime khac nhau, nen local venv co the khong duoc chia se.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess

BASE_DIR = '/content/drive/MyDrive/VKU/Graduation_project'
REPO_DIR = f'{BASE_DIR}/GradProject'
DRIVE_ROOT = f'{BASE_DIR}/GradProject_colab'
VENV_DIR = '/content/venvs/gradproject'
BOOTSTRAP_SCRIPT = f'{REPO_DIR}/colab/scripts/bootstrap_env.sh'

if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(
        'Khong tim thay repo. Hay chay clone_repo.ipynb truoc de tao '
        '/content/drive/MyDrive/VKU/Graduation_project/GradProject'
    )

if not os.path.isfile(BOOTSTRAP_SCRIPT):
    raise FileNotFoundError(f'Khong tim thay script bootstrap: {BOOTSTRAP_SCRIPT}')

# /content la local disk theo runtime; neu doi runtime thi venv se mat.
if not os.path.isfile(f'{VENV_DIR}/bin/python'):
    print('Local venv khong ton tai trong runtime hien tai. Dang chay bootstrap lai...')
    env = os.environ.copy()
    env['BASE_ROOT'] = BASE_DIR
    env['REPO_DIR'] = REPO_DIR
    env['DRIVE_ROOT'] = DRIVE_ROOT
    env['VENV_DIR'] = VENV_DIR
    env['ENABLE_VENV_SNAPSHOT'] = '1'
    env['FORCE_REINSTALL'] = '0'
    env['EXPECTED_PYTHON_MAJOR_MINOR'] = '3.11'
    subprocess.run(['bash', BOOTSTRAP_SCRIPT], check=True, env=env)

if not os.path.isfile(f'{VENV_DIR}/bin/python'):
    raise FileNotFoundError(
        f'Khong tim thay {VENV_DIR}/bin/python sau khi bootstrap.'
    )

print('Repo dir  :', REPO_DIR)
print('Drive root:', DRIVE_ROOT)
print('Venv dir  :', VENV_DIR)

In [ ]:
%%bash
set -euo pipefail
BASE_DIR=/content/drive/MyDrive/VKU/Graduation_project
export BASE_ROOT="$BASE_DIR"
export REPO_DIR="$BASE_DIR/GradProject"
export DRIVE_ROOT="$BASE_DIR/GradProject_colab"
export VENV_DIR="/content/venvs/gradproject"
export PORT=8000
chmod +x "$REPO_DIR/colab/scripts/run_server.sh"
"$REPO_DIR/colab/scripts/run_server.sh"

In [ ]:
%%bash
set -euo pipefail
BASE_DIR=/content/drive/MyDrive/VKU/Graduation_project
export BASE_ROOT="$BASE_DIR"
export REPO_DIR="$BASE_DIR/GradProject"
export DRIVE_ROOT="$BASE_DIR/GradProject_colab"
export PORT=8000

chmod +x "$REPO_DIR/colab/scripts/cloudflared_start.sh"
"$REPO_DIR/colab/scripts/cloudflared_start.sh"

In [ ]:
from pathlib import Path

base_dir = Path('/content/drive/MyDrive/VKU/Graduation_project')
p = base_dir / 'GradProject_colab' / 'runtime' / 'last_public_url.txt'
if p.exists():
    url = p.read_text().strip()
    print('HTTP URL:', url)
    print(url.replace('https://', 'wss://') + '/ws')
else:
    print('Public URL not found yet. Re-run tunnel cell or inspect tunnel.log.')

## Debug: Server Logs
Dung cell ben duoi de xem nhanh loi load model va traceback tu uvicorn.

In [ ]:
# %%bash
# set -euo pipefail
# BASE_DIR=/content/drive/MyDrive/VKU/Graduation_project
# LOG_FILE="$BASE_DIR/GradProject_colab/logs/server.log"
# echo "Server log file: $LOG_FILE"
# if [ -f "$LOG_FILE" ]; then
#   echo "----- last 200 lines -----"
#   tail -n 200 "$LOG_FILE"
#   echo
#   echo "----- error-focused grep -----"
#   grep -nE "ERROR|Traceback|ModelLoadError|ENGINE_FAILURE|Unable to open file|No such file|Exception" "$LOG_FILE" || true
# else
#   echo "[WARN] server.log not found yet"
# fi

In [ ]:
# %%bash
# set -euo pipefail
# BASE_DIR=/content/drive/MyDrive/VKU/Graduation_project
# LOG_FILE="$BASE_DIR/GradProject_colab/logs/server.log"
# echo "Following server.log (Ctrl+C de dung)..."
# tail -n 0 -f "$LOG_FILE"